# Mueller Matrix Visualization with Physical Realizability Filtering

In [10]:
# Standard imports
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

root_dir = Path().cwd().parent
sys.path.append(str(root_dir))

from src.utils.visualisation import visualize_save_mueller
from src.utils.file_paths import file_paths
from src.utils.pr_test import charpoly_vectorized


In [29]:
# Configuration
# file_path = '/Volumes/ep_ssd/horao/NPP /550/HT/2022-07-06_T_AUTOPSY-BF_FR_S1_1/polarimetry/550nm/MM.npz'

wavelength = '550'

figure_dir = Path('figures/npz_samples')
figure_dir.mkdir(parents=True, exist_ok=True)


In [30]:
# Load .npz file and extract Mueller matrix
with np.load(file_path) as data:
    mueller_matrix = data['nM']
    
print(f"Loaded Mueller matrix shape: {mueller_matrix.shape}")

# Auto-detect dimensions
num_rows, num_cols, num_channels = mueller_matrix.shape
print(f"Auto-detected dimensions: {num_rows} rows x {num_cols} cols x {num_channels} channels")


Loaded Mueller matrix shape: (388, 516, 16)
Auto-detected dimensions: 388 rows x 516 cols x 16 channels


In [31]:
# Compute physical realizability mask
def compute_pr_mask(mueller_matrix, batch_size=50000):
    H, W, _ = mueller_matrix.shape
    total_pixels = H * W
    
    mueller_matrices = mueller_matrix.reshape(-1, 4, 4)
    pr_results = np.zeros(total_pixels, dtype=bool)
    
    for i in range(0, total_pixels, batch_size):
        end_idx = min(i + batch_size, total_pixels)
        batch_matrices = mueller_matrices[i:end_idx]
        batch_results = charpoly_vectorized(batch_matrices)
        pr_results[i:end_idx] = batch_results
    
    pr_mask = pr_results.reshape(H, W)
    pr_pixels = np.sum(pr_mask)
    print(f"PR pixels: {pr_pixels:,} ({pr_pixels/total_pixels*100:.2f}%)")
    
    return pr_mask

pr_mask = compute_pr_mask(mueller_matrix)


PR pixels: 155,881 (77.86%)


In [32]:
# Apply PR filter
mueller_filtered = mueller_matrix.copy()
mueller_filtered[~pr_mask] = 0


In [ ]:
# Visualize original Mueller matrix
df_original = pd.DataFrame(mueller_matrix.reshape(-1, 16))

for offdiag_range in [(-0.1, 0.1), (-0.05, 0.05)]:
    range_label = f"original_offdiag{offdiag_range[0]}_{offdiag_range[1]}"
    visualize_save_mueller(
        data=df_original,
        visualisation_path=figure_dir,
        sample_number='',
        wavelength=wavelength,
        label=range_label,
        num_rows=num_rows,
        num_cols=num_cols,
        filter_zeros=False,
        show_fig=True,
        offdiagonal_range=offdiag_range,
    )


In [ ]:
# Visualize filtered Mueller matrix
df_filtered = pd.DataFrame(mueller_filtered.reshape(-1, 16))

for offdiag_range in [(-0.1, 0.1), (-0.05, 0.05)]:
    range_label = f"filtered_PR_offdiag{offdiag_range[0]}_{offdiag_range[1]}"
    visualize_save_mueller(
        data=df_filtered,
        visualisation_path=figure_dir,
        sample_number='',
        wavelength=wavelength,
        label=range_label,
        num_rows=num_rows,
        num_cols=num_cols,
        filter_zeros=True,
        show_fig=True,
        offdiagonal_range=offdiag_range,
    )
